### Import Libraries.

In [3]:
import pandas as pd 
import numpy as np 

In [4]:
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import RandomizedSearchCV

### Overview of data.

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv')

In [6]:
df.sample(5)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
213,0.14052,0.0,10.59,0,0.489,6.375,32.3,3.9454,4,277,18.6,385.81,9.38,28.1
224,0.31533,0.0,6.20,0,0.504,8.266,78.3,2.8944,8,307,17.4,385.05,4.14,44.8
380,88.97620,0.0,18.10,0,0.671,6.968,91.9,1.4165,24,666,20.2,396.90,17.21,10.4
126,0.38735,0.0,25.65,0,0.581,5.613,95.6,1.7572,2,188,19.1,359.29,27.26,15.7
354,0.04301,80.0,1.91,0,0.413,5.663,21.9,10.5857,4,334,22.0,382.80,8.05,18.2


In [7]:
# separate the feature and target. 
X = df.iloc[ : , : 13]
y = df['medv']

In [8]:
## Apply KNN Regressor without Grid Search CV.

In [12]:
# Model object. 
knn_model1 = KNeighborsRegressor()

# kfold object. 
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# training using cross validation. 
scores = cross_val_score(knn_model1, X, y, cv=kfold, scoring='r2')

# scores. 
print('Mean of all the scores : ', np.mean(scores))

Mean of all the scores :  0.5383871944766712


In [13]:
## Apply using Randomized Search CV. 

In [17]:
# Model object. 
knn_model2 = KNeighborsRegressor()

# create the parameter dictionary. 
param_dict = {
    'n_neighbors' : [1, 3, 5, 7, 9, 8, 11, 10, 17, 16, 19],
    'weights' : ['uniform', 'distance'], 
    'algorithm' : ['ball_tree', 'kd_tree', 'brute'], 
    'p' : [1, 2]
}


# finding best parameters using RandomizedSearchCV.
# here the iteration tells the number of random comination we take. 
rscv = RandomizedSearchCV(knn_model2, param_distributions=param_dict, n_iter=10, cv=kfold, refit=True, verbose=2, scoring='r2')

# training.
rscv.fit(X, y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END algorithm=kd_tree, n_neighbors=3, p=2, weights=distance; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=3, p=2, weights=distance; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=3, p=2, weights=distance; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=3, p=2, weights=distance; total time=   0.0s
[CV] END algorithm=kd_tree, n_neighbors=3, p=2, weights=distance; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=9, p=1, weights=uniform; total time=   0.1s
[CV] END algorithm=brute, n_neighbors=9, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=9, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=9, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=9, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=brute, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=bru

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=KNeighborsRegressor(),
                   param_distributions={'algorithm': ['ball_tree', 'kd_tree',
                                                      'brute'],
                                        'n_neighbors': [1, 3, 5, 7, 9, 8, 11,
                                                        10, 17, 16, 19],
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   scoring='r2', verbose=2)

In [19]:
# best estimator. 
rscv.best_estimator_

KNeighborsRegressor(algorithm='ball_tree', p=1, weights='distance')

In [20]:
# best score. 
rscv.best_score_

np.float64(0.6434974189445056)

In [21]:
# best parameters.
rscv.best_params_

{'weights': 'distance', 'p': 1, 'n_neighbors': 5, 'algorithm': 'ball_tree'}

In [22]:
# results of all the combination. 
all_results = pd.DataFrame(rscv.cv_results_)
all_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_weights,param_p,param_n_neighbors,param_algorithm,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004261,0.001076,0.004537,0.001691,distance,2,3,kd_tree,"{'weights': 'distance', 'p': 2, 'n_neighbors':...",0.695744,0.594466,0.400764,0.618602,0.457063,0.553328,0.108423,6
1,0.002232,0.000863,0.041114,0.073549,uniform,1,9,brute,"{'weights': 'uniform', 'p': 1, 'n_neighbors': ...",0.624180,0.613137,0.394960,0.605891,0.551363,0.557906,0.085252,5
2,0.001828,0.000211,0.004092,0.001226,uniform,1,1,brute,"{'weights': 'uniform', 'p': 1, 'n_neighbors': ...",0.530020,0.537940,0.451603,0.515197,0.504100,0.507772,0.030426,8
3,0.003094,0.000117,0.003063,0.000323,uniform,2,16,kd_tree,"{'weights': 'uniform', 'p': 2, 'n_neighbors': ...",0.498239,0.415235,0.331704,0.425135,0.358551,0.405773,0.057846,10
4,0.002612,0.000148,0.002985,0.000354,distance,2,5,ball_tree,"{'weights': 'distance', 'p': 2, 'n_neighbors':...",0.667400,0.639644,0.403622,0.622678,0.494980,0.565665,0.100327,4
5,0.002596,0.000148,0.002971,0.000070,distance,1,5,ball_tree,"{'weights': 'distance', 'p': 1, 'n_neighbors':...",0.713653,0.707998,0.473569,0.742470,0.579798,0.643497,0.101795,1
6,0.003211,0.000235,0.003060,0.000173,uniform,1,16,kd_tree,"{'weights': 'uniform', 'p': 1, 'n_neighbors': ...",0.584102,0.457590,0.377339,0.492575,0.407998,0.463921,0.072012,9
7,0.002648,0.000295,0.003047,0.000192,uniform,1,11,ball_tree,"{'weights': 'uniform', 'p': 1, 'n_neighbors': ...",0.623666,0.563247,0.395033,0.578496,0.499249,0.531938,0.079213,7
8,0.003422,0.000478,0.002964,0.000393,distance,1,16,kd_tree,"{'weights': 'distance', 'p': 1, 'n_neighbors':...",0.630221,0.616583,0.476724,0.608187,0.532832,0.572909,0.058812,2
9,0.002816,0.000102,0.002552,0.000071,uniform,1,8,kd_tree,"{'weights': 'uniform', 'p': 1, 'n_neighbors': ...",0.636695,0.619663,0.396544,0.623269,0.575916,0.570418,0.089301,3


In [23]:
# find most important features from the all results.
all_results[['param_algorithm',	'param_n_neighbors',	'param_p', 'param_weights', 'mean_test_score']].sort_values('mean_test_score',ascending=False)

,param_algorithm,param_n_neighbors,param_p,param_weights,mean_test_score
5,ball_tree,5,1,distance,0.643497
8,kd_tree,16,1,distance,0.572909
9,kd_tree,8,1,uniform,0.570418
4,ball_tree,5,2,distance,0.565665
1,brute,9,1,uniform,0.557906
0,kd_tree,3,2,distance,0.553328
7,ball_tree,11,1,uniform,0.531938
2,brute,1,1,uniform,0.507772
6,kd_tree,16,1,uniform,0.463921
3,kd_tree,16,2,uniform,0.405773


`NOTE : ` we can also using predict function we can predict on new data bcs at the last it will train new model for best parameters and store it in there object that are used for prediction in future.